# Regressão Logística (NumPy)

In [ ]:
import numpy as np
import pandas as pd
import pickle

from text_features import clean_text, TFIDFVectorizer, encode_labels, labels_to_onehot, decode_labels
from logistic_regression import LogisticRegression

In [ ]:
df_train = pd.read_csv('../datasets/dataset_train.csv', sep=';')
df_test  = pd.read_csv('../datasets/dataset_test.csv',  sep=';')
df_val   = pd.read_csv('../datasets/dataset_val.csv',   sep=';')

print(f'treino: {len(df_train)} | teste: {len(df_test)} | val: {len(df_val)}')
print('\ndistribuição de treino:')
print(df_train['Label'].value_counts())

## *pré-processamento*

In [ ]:
train_texts  = [clean_text(t) for t in df_train['Text'].tolist()]
train_labels = df_train['Label'].str.lower().tolist()

test_texts   = [clean_text(t) for t in df_test['Text'].tolist()]
test_labels  = df_test['Label'].str.lower().tolist()

# TF-IDF com vocabulário de 10000 palavras
vec     = TFIDFVectorizer(max_words=10000)
X_train = vec.fit_transform(train_texts)
X_test  = vec.transform(test_texts)

# labels em one-hot (necessário para a regressão logística NumPy)
Y_train = labels_to_onehot(encode_labels(train_labels))
Y_test  = labels_to_onehot(encode_labels(test_labels))

print(f'X_train: {X_train.shape} | Y_train: {Y_train.shape}')
print(f'X_test:  {X_test.shape}  | Y_test:  {Y_test.shape}')

## *treino*

In [ ]:
X_train_dense = X_train.toarray() if not isinstance(X_train,np.ndarray) else X_train

model = LogisticRegression(
    X=X_train_dense,
    y=Y_train,
    n_classes=5,
    standardize=False,
    l2_lambda=1e-4
)

model.build_model(alpha=0.1, iters=300, batch_size=32)

## *avaliação*

In [ ]:
X_train_dense = X_train.toarray() if not isinstance(X_train, np.ndarray) else X_train
X_test_dense  = X_test.toarray()  if not isinstance(X_test,  np.ndarray) else X_test

train_acc, train_f1 = model.score(X_train_dense, Y_train)
test_acc,  test_f1  = model.score(X_test_dense,  Y_test)

print(f'[TRAIN] accuracy={train_acc:.4f} | f1-macro={train_f1:.4f}')
print(f'[TEST]  accuracy={test_acc:.4f}  | f1-macro={test_f1:.4f}')

In [ ]:
model.save('../models/model_logreg.npz')

with open('../vectorizers/vectorizer_logreg.pkl', 'wb') as f:
    pickle.dump(vec, f)

print('modelo e vectorizer guardados.')

## *validação*

In [ ]:
val_texts  = [clean_text(t) for t in df_val['Text'].tolist()]
val_labels = df_val['Label'].str.lower().tolist()

X_val       = vec.transform(val_texts)
X_val_dense = X_val.toarray() if not isinstance(X_val, np.ndarray) else X_val
Y_val       = labels_to_onehot(encode_labels(val_labels))

val_acc, val_f1 = model.score(X_val_dense, Y_val)
print(f'accuracy={val_acc:.4f} | f1-macro={val_f1:.4f}')

preds_idx = [model.predict(x) for x in X_val_dense]
preds     = decode_labels(preds_idx)
print(pd.DataFrame({'ID': df_val['ID'], 'Predicted': preds, 'True': df_val['Label']}).to_string(index=False))